# PDF pipeline · IPC Gaza report, page 1

Run each cell in order. Each resolved piece exposes `textual_part` and `visual_part`. Its `source_ids` link to locations in `resolved_doc["source_blocks"]`.

Launch Jupyter with `OPENROUTER_API_KEY` in its environment to run Enrich and Embed. The README shows the Doppler command.


## Source

In [ ]:
import os
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

import pypdfium2
from IPython.display import Image, display

from bw_document_parser.embeddings import embed_document
from bw_document_parser.enrichment import enrich_document
from bw_document_parser.parsing import parse_pdf
from bw_document_parser.resolving import resolve_document
from bw_document_parser.search_entries import SearchEntry

filename = "IPC_Gaza_Strip_Acute_Food_Insecurity_Feb_July2024_Special_Brief.pdf"
pdf = next(path.resolve() for path in (Path(filename), Path("examples") / filename) if path.is_file())
scratch = TemporaryDirectory()
bundle = Path(scratch.name)

with pypdfium2.PdfDocument(str(pdf)) as source:
    with closing(source[0]) as page:
        with closing(page.render(scale=1.2)) as bitmap:
            display(bitmap.to_pil().copy())

## Parse
Docling gives the heading and paragraph their own source IDs and boxes.

In [ ]:
parsed_doc = parse_pdf(pdf, bundle, page_range=(1, 1))
blocks = {block["id"]: block for block in parsed_doc["pieces"]}
reading_order = parsed_doc["pages"][0]["piece_ids"]
overview_id = next(block_id for block_id in reading_order
                   if blocks[block_id]["content"].get("text", "").strip() == "Overview")
paragraph_id = reading_order[reading_order.index(overview_id) + 1]
display([{"id": block_id, "text": blocks[block_id]["content"]["text"],
          "locations": blocks[block_id]["locations"]}
         for block_id in (overview_id, paragraph_id)])

## Resolve
The Overview section becomes one piece. Its blocks still point to their original positions.

In [ ]:
resolved_doc = resolve_document(parsed_doc)
pieces = resolved_doc["pages"][0]["pieces"]

def section_for(doc, block_id):
    return next(piece for page in doc["pages"] for piece in page["pieces"]
                if block_id in piece["source_ids"])

overview = section_for(resolved_doc, overview_id)
assert paragraph_id in overview["source_ids"]
display(overview)

visuals = [image for piece in pieces for image in piece["visual_part"] or []]
visual = next((image for image in visuals if image["source_id"] == "p1:#/pictures/2"), visuals[0])
figure_id = visual["source_id"]
figure_path = bundle / visual["url"]
display(visual)
display(Image(filename=str(figure_path), width=430))

In [ ]:
clean_pieces = [
    {
        key: piece[key]
        for key in ("title", "textual_part", "visual_part")
        if piece.get(key)
    }
    for piece in pieces
]
clean_pieces

## Enrich
Decider checks the selected figure. Approved images get a description. Skipped images are excluded from the final piece.

In [ ]:
if not os.getenv("OPENROUTER_API_KEY"):
    raise RuntimeError("Launch Jupyter through Doppler or set OPENROUTER_API_KEY in its environment.")

enriched_doc, complete = enrich_document(resolved_doc, bundle, block_ids=[figure_id], workers=1)
if not complete:
    raise RuntimeError("Figure enrichment failed. See the provider error above and rerun this cell.")

figure_piece = section_for(enriched_doc, figure_id)
enriched_visual = next((image for image in figure_piece["visual_part"] or [] if image["source_id"] == figure_id), None)
display(enriched_doc["source_blocks"][figure_id]["enrichment_decision"])
if enriched_visual is not None:
    display(Image(filename=str(figure_path), width=430))
    display(enriched_visual)

## Embed
Send the Overview heading and paragraph together as one text input.

In [ ]:
overview_blocks = [resolved_doc["source_blocks"][i] for i in (overview_id, paragraph_id)]
target = SearchEntry(
    id=f"{overview_id}:heading-and-paragraph",
    piece_ids=[overview_id, paragraph_id],
    locations=[location for block in overview_blocks for location in block["locations"]],
    text="\n\n".join(block["content"]["text"] for block in overview_blocks),
)
embedded_doc, complete = embed_document(enriched_doc, bundle, "text", targets=[target], workers=1)
if not complete:
    raise RuntimeError("Embedding failed. See the provider error above and rerun this cell.")

embedded_piece = section_for(embedded_doc, overview_id)
record = next(entry for entry in embedded_piece["embeddings"] if entry["id"] == target.id)
display({
    "piece_id": embedded_piece["id"], "target": record["piece_ids"], "model": record["model"],
    "length": len(record["vector"]), "first_values": record["vector"][:5],
})

def structure(doc):
    return [(piece["id"], piece["source_ids"])
            for page in doc["pages"] for piece in page["pieces"]]

assert structure(resolved_doc) == structure(enriched_doc) == structure(embedded_doc)